# Анализ данных GenUI

Цель анализа — проверить качество разметки, описать состав пар интерфейсов и обосновать подготовку данных для попарного ранжирования. Источник: SALT-NLP/GenUI, ревизия `11c9182f5f1452f28a8714bf3dea6fbd218963c7`. Лицензия: CC BY-NC 4.0.

Одна обучающая строка соответствует паре интерфейсов, созданных по одному запросу. Общий победитель определяется строгим большинством человеческих оценок. Размеры и структура HTML рассматриваются как признаки; страницы и JavaScript в процессе анализа не исполняются.

Запуск из корня проекта: `make ml-up`, затем `make eda`. Выполненный ноутбук, таблицы, графики и отчёт сохраняются в MLflow и в `artifacts/mlflow/eda/`.

In [ ]:
%matplotlib inline
import hashlib
import json
import os
from pathlib import Path

import mlflow
import pandas as pd
from IPython.display import Markdown, display

from genui_quality_gate.ml.data import DATASET_SOURCE, load_dataset, split_dataset
from genui_quality_gate.ml.eda import SOURCE_LABELS, build_analysis, log_analysis
from genui_quality_gate.ml.train import log_dataset

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
OUTPUT = Path(os.getenv("GENUI_EDA_OUTPUT_DIR", str(ROOT / "artifacts/mlflow/eda")))
OUTPUT.mkdir(parents=True, exist_ok=True)
frame, manifest = load_dataset(ROOT / "data/genui")
splits = split_dataset(frame)
analysis = build_analysis(frame, splits, manifest)

## 1. Исходная разметка и фильтрация

Проверяются число оценок на пару, обязательные поля и служебные флаги. Повторные оценки одного участника для одной пары исключаются при агрегации. Пары без строгого большинства не используются для обучения: случайный выбор победителя добавил бы шум в целевую переменную. Ничьи при этом остаются в описании исходных данных.

In [ ]:
display(pd.Series(manifest["counts"], name="count").to_frame())
display(pd.Series(manifest["annotations"]["missing_required_values"], name="missing").to_frame())
display(pd.Series(manifest["annotations"]["flags"], name="flagged").to_frame())
display(
    pd.Series(manifest["annotations"]["ratings_per_pair"], name="pairs")
    .rename_axis("ratings_per_pair")
    .to_frame()
)
counts = manifest["counts"]
display(
    Markdown(
        f"После агрегации осталось **{counts['retained_pairs']} из {counts['total_pairs']} пар**. Исключено {counts.get('excluded_no_majority_pairs', 0)} пар без большинства. Число участников разметки: {manifest['annotations']['annotators']}."
    )
)

## 2. Победители и согласие оценщиков

После агрегации порядок A/B определяется детерминированно по хешу пары. Это уменьшает зависимость целевой переменной от расположения конкретного генератора в исходной анкете. `target=1` означает победу A. Согласие оценщиков — доля голосов за победителя среди всех голосов пары, включая ничьи.

Исходные A/B в правой части графика относятся к расположению интерфейсов в анкете и не являются названиями моделей.

In [ ]:
display(analysis.figures["distributions"])
display(
    frame.groupby("target").agg(pairs=("pair_id", "size"), mean_agreement=("agreement", "mean"))
)
display(frame["agreement"].value_counts().sort_index().rename("pairs").to_frame())
display(
    Markdown(
        f"Доля побед A составляет {frame['target'].mean():.1%}. Доля пар с единогласным выбором — {(frame['agreement'] == 1).mean():.1%}. Для сравнения моделей используется balanced accuracy; порог решения фиксирован на 0.5."
    )
)

## 3. Отдельные критерии пользовательской оценки

Разметка содержит семь критериев: соответствие запросу, эффективность выполнения задачи, удобство, обучаемость, понятность информации, визуальную привлекательность и удовлетворённость взаимодействием.

Для каждого критерия рассматриваются доли A/B/ничья и согласие с общим выбором. Последняя величина рассчитывается только для оценок без ничьей в обоих полях. Она описывает связь ответов одного оценщика и не показывает причинное влияние критерия на общий выбор.

Отдельно рассчитывается средняя доля наиболее частого ответа в каждой паре после приведения A/B к одному порядку. Это согласие между оценщиками. Единогласная ничья учитывается как согласие, но не задаёт победителя; поэтому в таблице также приведена доля пар без большинства за A или B.

In [ ]:
display(analysis.tables["criteria"].round(4))
display(analysis.figures["annotations"])
criteria = analysis.tables["criteria"]
most_ties = criteria.loc[criteria["tie_share"].idxmax()]
closest = criteria.loc[criteria["overall_agreement"].idxmax()]
display(
    Markdown(
        f"Наибольшая доля ничьих наблюдается для `{most_ties['criterion']}`: {most_ties['tie_share']:.1%}. Наибольшее согласие с общим победителем — для `{closest['criterion']}`: {closest['overall_agreement']:.1%}. Для основной задачи используется overallWinner; отдельные критерии в целевую переменную не подмешиваются."
    )
)

## 4. Покрытие запросов и интерфейсов

Один интерфейс может участвовать в нескольких парах. Поэтому число строк не равно числу независимых документов. Проверяется, сколько пар осталось для каждого запроса и как представлены источники интерфейсов после фильтрации.

Сравнения ограничены схемой исходного исследования: в наборе пар представлен один общий сравниваемый источник. Доли побед генераторов в таком наборе нельзя считать универсальным рейтингом. Имена генераторов не передаются в признаки модели.

In [ ]:
display(analysis.tables["source_coverage"].rename(index=SOURCE_LABELS))
display(analysis.tables["prompt_coverage"].describe())
display(analysis.figures["coverage"])
quality = analysis.summary["quality"]
display(
    Markdown(
        f"В исходной ревизии загружено {manifest['source_html_files']} HTML-файлов. В сохранённых парах используются {quality['unique_documents']} документов для {analysis.summary['prompts']} запросов. Число пар на запрос варьируется от {analysis.summary['pairs_per_prompt_min']} до {analysis.summary['pairs_per_prompt_max']}."
    )
)

## 5. Качество и размеры HTML

Проверяются пропуски, пустые HTML, повторные идентификаторы пар и точные совпадения содержимого документов. SHA256 позволяет различать повторное использование файла и одинаковое содержимое разных файлов. Размеры HTML рассчитываются по уникальным документам, размеры запросов — по уникальным запросам.

Большие значения счётчиков преобразуются через log1p. Длина HTML не рассматривается как самостоятельная оценка качества: длинная страница может содержать служебный код или повторяющуюся разметку.

Для пар с полностью одинаковым HTML доступные модели признаки совпадают. Такой вход не позволяет объяснить размеченное предпочтение. Случай сохраняется в исходном benchmark и отмечается как ограничение; правила фильтрации не меняются по результатам test.

In [ ]:
display(pd.Series(quality, name="value").to_frame())
display(
    analysis.tables["documents"]["characters"]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .rename("characters")
    .to_frame()
)
display(analysis.figures["documents"])

## 6. Разбиение и проверка пересечений

Train/validation/test формируются в пропорции 60/20/20 по запросам. При случайном разбиении строк разные пары одного запроса могли бы попасть в разные выборки. Проверяются пересечения запросов, путей документов и SHA256 HTML.

Зеркальные пары добавляются только в train после разбиения. Validation используется для выбора варианта модели, test — для итоговой оценки выбранного варианта.

In [ ]:
display(analysis.tables["splits"].round(4))
display(analysis.tables["split_overlap"])
display(analysis.figures["splits"])
overlap = analysis.tables["split_overlap"]
assert (overlap["prompt_overlap"] == 0).all()
display(
    Markdown(
        f"Пересечений запросов: {int(overlap['prompt_overlap'].sum())}; документов: {int(overlap['document_overlap'].sum())}; содержимого HTML: {int(overlap['html_digest_overlap'].sum())}. Проверка касается точных совпадений и не исключает похожие шаблоны или семантически близкие запросы."
    )
)

## 7. HTML-признаки на train

Связь признаков с предпочтением анализируется только на исходном train, до зеркалирования. В каждой паре сравниваются значения признаков победителя и проигравшего. Отдельно показаны доли положительного, нулевого и отрицательного различия.

Это описательная проверка выбранных признаков. Пары внутри одного запроса зависимы, поэтому доли не сопровождаются выводом о статистической значимости. Accessibility-признаки учитывают наличие атрибутов и элементов, но не заменяют полноценную проверку доступности.

In [ ]:
display(analysis.tables["training_feature_statistics"].round(3))
display(analysis.tables["feature_preferences"].round(4))
display(analysis.figures["feature_preferences"])

## 8. Корреляции признаков

Корреляции Spearman рассчитаны по уникальным документам train. Счётчики длины, текста и числа тегов могут описывать близкие свойства страницы. Для логистической регрессии используются масштабирование и регуляризация. Корреляция признаков не показывает их отдельный вклад в качество модели.

In [ ]:
display(analysis.figures["feature_correlations"])

## 9. Выводы и ограничения

Для обучения используются пары со строгим большинством голосов. Разбиение выполняется по запросам; баланс train обеспечивается зеркалированием пар. Сравниваются baseline, структурные HTML-признаки и расширенный вариант с текстовыми и accessibility-признаками. Основная метрика — balanced accuracy; дополнительно сохраняются ROC AUC, F1, log loss и Brier score.

Датасет небольшой, а пары одного запроса зависимы. Схема сравнений и исключение спорных пар ограничивают переносимость результатов. Текущие признаки не оценивают скриншоты, фактическое поведение JavaScript и внешний вид после загрузки ресурсов. Поэтому результат представляет исследовательский прототип, а не проверенную оценку качества любого веб-интерфейса.

Графики, таблицы, отчёт и выполненный ноутбук сохраняются в отдельном EDA Run. Этот Run связан с датасетом через source и digest; обучающая часть анализа отдельно отмечена как `eda_training`.

In [ ]:
for name, figure in analysis.figures.items():
    figure.savefig(OUTPUT / f"{name}.png", dpi=150, bbox_inches="tight")
(OUTPUT / "tables").mkdir(exist_ok=True)
for name, table in analysis.tables.items():
    table.to_csv(OUTPUT / "tables" / f"{name}.csv")
(OUTPUT / "report.md").write_text(analysis.report)
(OUTPUT / "summary.json").write_text(json.dumps(analysis.summary, indent=2, ensure_ascii=False))
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5001"))
mlflow.set_experiment("genui-quality")
ml_sources = sorted((ROOT / "src/genui_quality_gate/ml").glob("*.py"))
with mlflow.start_run(run_name="eda-detailed") as run:
    mlflow.set_tags(
        {
            "kind": "eda",
            "dataset_revision": manifest["revision"],
            "dataset_digest": analysis.summary["dataset_digest"],
            "feature_analysis_scope": "original training split only",
            "ml_source_sha256": hashlib.sha256(
                b"".join(path.read_bytes() for path in ml_sources)
            ).hexdigest(),
            "mlflow.source.name": "notebooks/eda.ipynb",
            "source_notebook_sha256": hashlib.sha256(
                (ROOT / "notebooks/eda.ipynb").read_bytes()
            ).hexdigest(),
        }
    )
    log_dataset(frame, DATASET_SOURCE, "eda")
    log_dataset(splits["train"], DATASET_SOURCE, "eda_training")
    log_analysis(analysis)
    mlflow.log_artifact(str(ROOT / "notebooks/eda.ipynb"), artifact_path="eda/source")
    for path in ml_sources:
        mlflow.log_artifact(str(path), artifact_path="eda/source/ml")
    result = {"run_id": run.info.run_id, "dataset_digest": analysis.summary["dataset_digest"]}
(OUTPUT / "run.json").write_text(json.dumps(result, indent=2))
display(result)